# Module 2 — Feature Engineering

This notebook transforms raw box-score columns into model-ready features, with one non-negotiable rule: every feature may only use information available before the game it is being used to predict. Breaking this rule produces target leakage, the single worst failure mode in machine learning, because the model looks accurate during testing and silently collapses in production once the leaked future information no longer exists.

Every rolling feature uses the pattern `.shift(1).rolling(N)` inside a `groupby('PLAYER_ID').transform(...)` call, which excludes the current row from its own window and preserves per-player boundaries.

**Final feature set (9 features):** `rest_days_capped`, `returned_from_absence`, `rolling_fp_5`, `rolling_fp_10`, `rolling_min_10`, `rolling_fp_std_10`, `fp_momentum`, `is_home`, `opp_fp_allowed`.

**Output:** `data/processed/model_df.parquet` with features appended.

## Setup — load data and recompute FP

In [85]:
import pandas as pd
import numpy as np

# Load the raw player-games data from Module 1's data pull
pg = pd.read_parquet('data/raw/player_games_2021_2024.parquet')

# Recompute FP using user_league scoring (same formula as Module 1 Q6)
pg['FP'] = (
    pg['PTS']  *  1 +
    pg['FGM']  *  2 +
    pg['FGA']  * -1 +
    pg['FTM']  *  1 +
    pg['FTA']  * -1 +
    pg['FG3M'] *  1 +
    pg['REB']  *  1 +
    pg['AST']  *  2 +
    pg['STL']  *  4 +
    pg['BLK']  *  4 +
    pg['TOV']  * -2
)

print(f"Loaded: {pg.shape[0]:,} rows x {pg.shape[1]} cols")
print(f"FP range: {pg['FP'].min():.1f} to {pg['FP'].max():.1f}")

Loaded: 104,641 rows x 34 cols
FP range: -17.0 to 119.0


## Feature 1: rest_days (raw)

**What it is:** days since this player's last game.

**Why it matters:**
- 1 day → back-to-back, fatigue
- 2-3 days → normal
- 4-6 days → ideal rest
- 30+ days → injury return, likely rusty

**Pandas:** same pattern as Q5 — sort by (PLAYER_ID, GAME_DATE), then `.groupby('PLAYER_ID')['GAME_DATE'].diff().dt.days`.

**Type this yourself** (new 70/30 rule — simple one-liners come from your fingers):

In [86]:
# Type this yourself — do not copy-paste
pg['rest_days']=pg.sort_values(['PLAYER_ID','GAME_DATE']).groupby('PLAYER_ID')['GAME_DATE'].diff().dt.days
pg['rest_days']

0         NaN
1         NaN
2         NaN
3         NaN
4         NaN
         ... 
104636    2.0
104637    2.0
104638    2.0
104639    2.0
104640    2.0
Name: rest_days, Length: 104641, dtype: float64

In [87]:
# Check the distribution
pg['rest_days'].describe()

count    103716.000000
mean          6.017249
std          27.476793
min           1.000000
25%           2.000000
50%           2.000000
75%           3.000000
max        1170.000000
Name: rest_days, dtype: float64

## Feature decomposition — one noisy column → two clean signals

The describe showed mean (6) ≫ median (2) and max of 1170 days. One column was hiding two completely different real-world situations:

- `rest_days = 2` → fatigue signal (tired from last game)
- `rest_days = 180` → fresh/offseason/injury return (totally different thing)

**Pro move:** split into two features so the model learns each signal cleanly.

| New feature | What it captures | Formula |
|---|---|---|
| `rest_days_capped` | Fatigue (back-to-back vs normal rest) | `rest_days` clipped at max 10 |
| `returned_from_absence` | Injury returns + season openers | 1 if `rest_days >= 30`, else 0 |

**Threshold = 30 days** — matches our Module 1 Q5 definition of a "mid-season gap." Consistency matters.

### 1a. `rest_days_capped` — fatigue signal

Type it yourself. `.clip(upper=10)` means "any value above 10 becomes 10."

In [88]:
# Type this yourself
pg['rest_days_capped']=pg['rest_days'].clip(upper=10)


In [89]:
# Inspect
pg['rest_days_capped'].describe()

count    103716.000000
mean          2.784276
std           2.122438
min           1.000000
25%           2.000000
50%           2.000000
75%           3.000000
max          10.000000
Name: rest_days_capped, dtype: float64

### 1b. `returned_from_absence` — injury/season-opener flag

Type it yourself. The expression `(pg['rest_days'] >= 30)` creates a boolean Series (True/False), and `.astype(int)` converts it to 1/0.

In [90]:
# Type this yourself
pg['returned_from_absence']=pg['rest_days'] >= 30

In [91]:
# How many games were "returns from absence"?
pg['returned_from_absence'].value_counts()

returned_from_absence
False    102636
True       2005
Name: count, dtype: int64

## Feature 2: `rolling_fp_5` — recent form (THE big one)

**What it is:** average FP over this player's **previous 5 games** (not including today).

**Why it matters:** best single predictor of today's FP = how the player has been performing lately.

**The leakage trap:** if game 10's rolling_fp_5 includes game 10's own FP, we leaked the answer. Must be games 5-9.

**The pandas pattern (burn this into memory):**
```
sort → groupby → shift(1) → rolling(N) → mean()
```

- `sort_values(['PLAYER_ID', 'GAME_DATE'])` → put each player's games in chronological order
- `.groupby('PLAYER_ID')` → compute separately per player (don't average across players!)
- `.shift(1)` → push every FP down one row → today's row now holds yesterday's FP
- `.rolling(5)` → take a 5-game window
- `.mean()` → average that window

Order matters. `shift(1)` BEFORE `rolling(5)` is what prevents leakage.

In [92]:
# FIX: sort pg in place, then use transform() so .shift(1) and .rolling(5)
# BOTH happen INSIDE each player's group (no cross-player bleed).

pg = pg.sort_values(['PLAYER_ID', 'GAME_DATE']).reset_index(drop=True)

pg['rolling_fp_5'] = pg.groupby('PLAYER_ID')['FP'].transform(
    lambda s: s.shift(1).rolling(5).mean()
)

pg['rolling_fp_5'].describe()

count    100247.000000
mean         22.387788
std          13.131053
min          -2.000000
25%          12.400000
50%          20.800000
75%          30.600000
max          88.400000
Name: rolling_fp_5, dtype: float64

In [93]:
# Sanity check: look at LeBron's first 15 games. rolling_fp_5 should trail FP by one game,
# first ~5 rows should be NaN (not enough history), then populate with the trailing mean.

(pg[pg['PLAYER_NAME'] == 'LeBron James']
   .sort_values('GAME_DATE')
   [['GAME_DATE', 'FP', 'rolling_fp_5']]
   .head(15))

,GAME_DATE,FP,rolling_fp_5
1,2021-10-19,60,NaN
2,2021-10-22,38,NaN
3,2021-10-24,45,NaN
4,2021-10-29,46,NaN
5,2021-10-31,46,NaN
6,2021-11-02,61,47.0
7,2021-11-19,41,47.2
8,2021-11-21,19,47.8
9,2021-11-24,65,42.6
10,2021-11-26,43,46.4


In [94]:
pg['rolling_fp_10'] = pg.groupby('PLAYER_ID')['FP'].transform(
    lambda s: s.shift(1).rolling(10).mean()
)

In [95]:
pg[['rolling_fp_5', 'rolling_fp_10']].describe()

,rolling_fp_5,rolling_fp_10
count,100247.000000,96263.000000
mean,22.387788,22.670065
std,13.131053,12.412738
min,-2.000000,-0.600000
25%,12.400000,13.200000
50%,20.800000,21.000000
75%,30.600000,30.500000
max,88.400000,77.500000


In [96]:
pg['is_home'] = pg['MATCHUP'].str.contains('vs.').astype(int)
pg['is_home'].value_counts()

is_home
1    52438
0    52203
Name: count, dtype: int64

## Feature 5: `opp_fp_allowed` — opponent defensive strength

**Idea:** Current features describe the player. This one describes the OPPONENT. A player facing elite defense (Boston) should score less than one facing a bad defense (Washington).

**4-step pipeline:**
1. Extract opponent team abbreviation from MATCHUP
2. Sum FP per (opponent, date) → team's FP-allowed-per-game
3. Rolling 10-game avg per team (leakage-safe shift+rolling)
4. Merge back to pg by (opponent, date)

**New concept introduced:** `.merge()` — joining two DataFrames. Like VLOOKUP in Excel.

In [97]:
# Step 1: Extract opponent abbreviation from MATCHUP.
# 'LAL vs. BOS' → split by space → ['LAL', 'vs.', 'BOS'] → take last element → 'BOS'
# 'LAL @ BOS'   → split by space → ['LAL', '@', 'BOS']   → take last element → 'BOS'

pg['OPP'] = pg['MATCHUP'].str.split(' ').str[-1]
pg[['MATCHUP', 'OPP']].head()

,MATCHUP,OPP
0,BOS vs. CLE,CLE
1,LAL vs. GSW,GSW
2,LAL vs. PHX,PHX
3,LAL vs. MEM,MEM
4,LAL vs. CLE,CLE


In [98]:
# Step 2: For each (OPP, GAME_DATE), sum all players' FP.
# Tells us: 'In this game, team X allowed Y total FP to its opponent.'
# Result: one row per (team, date), with FP_ALLOWED column.

team_game_fp = (
    pg.groupby(['OPP', 'GAME_DATE'])['FP'].sum()
      .reset_index()
      .rename(columns={'OPP': 'TEAM', 'FP': 'FP_ALLOWED'})
)

print(f'Shape: {team_game_fp.shape}')
team_game_fp.head()

Shape: (9840, 3)


,TEAM,GAME_DATE,FP_ALLOWED
0,ATL,2021-10-21,160
1,ATL,2021-10-23,214
2,ATL,2021-10-25,211
3,ATL,2021-10-27,214
4,ATL,2021-10-28,255


In [99]:
# Step 3: Per-team rolling 10-game avg of FP_ALLOWED (leakage-safe: shift(1) BEFORE rolling).
# Same pattern as rolling_fp_5/10, but applied to team-level data.

team_game_fp = team_game_fp.sort_values(['TEAM', 'GAME_DATE']).reset_index(drop=True)
team_game_fp['FP_ALLOWED_ROLLING_10'] = (
    team_game_fp.groupby('TEAM')['FP_ALLOWED']
                .transform(lambda s: s.shift(1).rolling(10).mean())
)

team_game_fp.head(15)

,TEAM,GAME_DATE,FP_ALLOWED,FP_ALLOWED_ROLLING_10
0,ATL,2021-10-21,160,NaN
1,ATL,2021-10-23,214,NaN
2,ATL,2021-10-25,211,NaN
3,ATL,2021-10-27,214,NaN
4,ATL,2021-10-28,255,NaN
5,ATL,2021-10-30,268,NaN
6,ATL,2021-11-01,230,NaN
7,ATL,2021-11-03,276,NaN
8,ATL,2021-11-04,246,NaN
9,ATL,2021-11-06,259,NaN


In [100]:
# Step 4: Merge the team rolling stat back onto pg using (OPP, GAME_DATE).
# Each player's row now has: opp_fp_allowed = how much FP their opponent has been giving up lately.

pg = pg.merge(
    team_game_fp[['TEAM', 'GAME_DATE', 'FP_ALLOWED_ROLLING_10']]
      .rename(columns={'TEAM': 'OPP', 'FP_ALLOWED_ROLLING_10': 'opp_fp_allowed'}),
    on=['OPP', 'GAME_DATE'],
    how='left'
)

pg['opp_fp_allowed'].describe()

count    101346.000000
mean        235.020511
std          17.636728
min         181.200000
25%         222.500000
50%         234.200000
75%         246.400000
max         301.700000
Name: opp_fp_allowed, dtype: float64

In [101]:
# Sanity check: which teams are the stingiest overall?  Which are the most generous?

team_defense = (
    pg.groupby('OPP')['opp_fp_allowed'].mean()
      .sort_values()
)

print('TOP 5 BEST DEFENSES (fewest FP allowed per game):')
print(team_defense.head(5).round(1))
print('\nTOP 5 WORST DEFENSES (most FP allowed per game):')
print(team_defense.tail(5).round(1))

TOP 5 BEST DEFENSES (fewest FP allowed per game):
OPP
BOS    216.3
CLE    222.8
MIA    223.6
NYK    224.6
MIN    226.1
Name: opp_fp_allowed, dtype: float64

TOP 5 WORST DEFENSES (most FP allowed per game):
OPP
ATL    246.1
POR    246.8
WAS    248.4
SAS    248.4
UTA    250.2
Name: opp_fp_allowed, dtype: float64


In [102]:
pg['rolling_min_10'] = pg.groupby('PLAYER_ID')['MIN'].transform(
    lambda s: s.shift(1).rolling(10).mean()
)

## Feature 7: `rolling_fp_std_10` — volatility signal

Same pattern as `rolling_fp_10`, but `.std()` instead of `.mean()`. Captures how CONSISTENT a player is — low std = predictable, high std = boom-or-bust.

In [103]:
pg['rolling_fp_std_10'] = pg.groupby('PLAYER_ID')['FP'].transform(
    lambda s: s.shift(1).rolling(10).std()
)
pg['rolling_fp_std_10'].describe()

count    96263.000000
mean        11.507413
std          3.835022
min          0.875595
25%          8.832390
50%         11.318127
75%         13.926793
max         32.121471
Name: rolling_fp_std_10, dtype: float64

## Feature 8: `fp_momentum` — explicit trend signal

`rolling_fp_5 - rolling_fp_10`. Positive = recent hot streak. Negative = cold streak. Gives trees an explicit trend variable to split on.

In [104]:
pg['fp_momentum'] = pg['rolling_fp_5'] - pg['rolling_fp_10']
pg['fp_momentum'].describe()

count    96263.000000
mean         0.044379
std          4.161571
min        -20.500000
25%         -2.600000
50%          0.000000
75%          2.700000
max         20.500000
Name: fp_momentum, dtype: float64

## Module 2 complete — save the feature-engineered dataset

We now have:
- **Target:** `FP`
- **Features:** `rest_days_capped`, `returned_from_absence`, `rolling_fp_5`, `rolling_fp_10`, `is_home`, `opp_fp_allowed`
- **Metadata (for later):** `PLAYER_ID`, `PLAYER_NAME`, `GAME_DATE`, `SEASON_YEAR`, `TEAM_ABBREVIATION`, `OPP`

Save to parquet so Module 3 can load the finished product.

In [105]:
feature_cols = ['rest_days_capped', 'returned_from_absence',
                'rolling_fp_5', 'rolling_fp_10',
                'is_home', 'opp_fp_allowed', 'rolling_min_10',
                'rolling_fp_std_10', 'fp_momentum']
meta_cols    = ['PLAYER_ID', 'PLAYER_NAME', 'GAME_DATE',
                'TEAM_ABBREVIATION', 'OPP']
target_col   = ['FP']

model_df = pg[meta_cols + feature_cols + target_col].copy()

import os
os.makedirs('data/processed', exist_ok=True)
model_df.to_parquet('data/processed/model_df.parquet', index=False)

print(f'Saved: {model_df.shape[0]:,} rows x {model_df.shape[1]} cols')
print(f'Features: {feature_cols}')
print(f'Target:   {target_col}')
model_df.head()

Saved: 104,641 rows x 15 cols
Features: ['rest_days_capped', 'returned_from_absence', 'rolling_fp_5', 'rolling_fp_10', 'is_home', 'opp_fp_allowed', 'rolling_min_10', 'rolling_fp_std_10', 'fp_momentum']
Target:   ['FP']


,PLAYER_ID,PLAYER_NAME,GAME_DATE,TEAM_ABBREVIATION,OPP,rest_days_capped,returned_from_absence,rolling_fp_5,rolling_fp_10,is_home,opp_fp_allowed,rolling_min_10,rolling_fp_std_10,fp_momentum,FP
0,2207,Joe Johnson,2021-12-22,BOS,CLE,NaN,False,NaN,NaN,1,191.9,NaN,NaN,NaN,3
1,2544,LeBron James,2021-10-19,LAL,GSW,NaN,False,NaN,NaN,1,NaN,NaN,NaN,NaN,60
2,2544,LeBron James,2021-10-22,LAL,PHX,3.0,False,NaN,NaN,1,NaN,NaN,NaN,NaN,38
3,2544,LeBron James,2021-10-24,LAL,MEM,2.0,False,NaN,NaN,1,NaN,NaN,NaN,NaN,45
4,2544,LeBron James,2021-10-29,LAL,CLE,5.0,False,NaN,NaN,1,NaN,NaN,NaN,NaN,46
